In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import ifft
import sys, os
from pathlib import Path

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.simulation.mixing_model import (
    construct_geometry,
    select_active_sources,
    generate_signals,
    compute_delay_matrix,
    simulate,
    quick_sim,
    quick_sector_sim,
)
from cs_priors.plotting.plot_complex import plot_complex_matrix_on_ax, plot_matrices
from cs_priors.plotting.plotting import plot_equation, plot_waveform_column_auto, plot_overview
from cs_priors.plotting.plot_geometry import plot_mics, plot_sources, plot_geometry_on_ax
from cs_priors.plotting.plot_signals import plot_signals, plot_magnitude_spectrum, plot_recovery
from cs_priors.plotting.plot_simulation import plot_simulation_report

# Simulation Demo
Vi har fire forbedringer til simuleringen
1. Samplingsraten, signallengden og signalet kan defineres uavhengig av de andre parameterene
2. Simuleringen har en metode for hvit støy, standard normalfordelt randn
3. Støy på målingene $Y = AX +\varepsilon$ 
4. Sektorsimulering

## 1. Til sammenligning: Vanlig oppsett med sinuser og hvordan frekvensdomenet vil se ut

In [ ]:
# Parameters
M = 6           # microphones
S = 5           # candidate sources
num_active = 2  # active (non-mute) sources
fs = 2000.0     # sampling rate [Hz]
duration = 0.05 # 50 ms
seed = 42

# 1. Geometry
mics, sources = construct_geometry(
    num_mics=M,
    array_type="circular",
    mic_spacing=0.1,
    num_sources=S,
    source_distance=1.5,
    angle_base=np.pi / 4,
    angle_step=0.3,
)

# 2. Select active sources
active = select_active_sources(sources, num_active, seed=seed)
print(f"Active source indices: {active}")

# 3. Generate signals (sum-of-sines)
frequencies = [[200, 300], [150, 250], [300], [100, 400], [100]]
phases = [0.0, 0.3, 0.6, 0.9, 1.2]
sources, t = generate_signals(
    sources, active, fs, duration,
    mode="sine", frequencies=frequencies, phases=phases,
)

# 4. Simulate  Y = A @ X
sim = simulate(mics, sources, active, fs, duration)
print(f"Shapes — x: {sim.x.shape}, X: {sim.X.shape}, A: {sim.A.shape}, Y: {sim.Y.shape}")

## Geometrien

In [ ]:
for i in active:
    sources[i].label = "active"
fig, ax = plt.subplots(figsize=(6, 6))
plot_geometry_on_ax(ax,sim.mics,sources, pad_factor=0.1)
plt.show()

## 3. Tid- og frekvensdomenet for kildesignalet $x$

In [ ]:
plt.rcParams['figure.dpi'] = 150
plot_signals(sim.x)
plot_magnitude_spectrum(sim.X, sim.freqs)
plt.show()

# Response $y$

In [ ]:
y = ifft(sim.Y).real
plot_signals(y)
plot_magnitude_spectrum(sim.Y, sim.freqs)
plt.show()

# 4. White noise

In [ ]:
sim = quick_sim(num_mics=5, num_sources=3, num_active=2, sampling_rate=2000)
plot_signals(sim.x)
plot_magnitude_spectrum(sim.X, sim.freqs)

## 7. Pseudoinverse recovery (overdetermined: M > S)

In [ ]:

plot_recovery(sim, sim.X_pinv, dpi=100)
plot_matrices([sim.X, sim.X_pinv],titles=["X_true", "X_pinv"], show_values=False, dpi=100)
plt.show()

# Pseudoinverse underdetermined for white noise

In [ ]:
from cs_priors.solvers.group_lasso import group_lasso_solve, no_groups, real_imag_groups, frequency_groups, random_groups
num_mics = 10
num_sources = 50
num_active = 2
duration = 0.01
sampling_rate = 2000
max_iter=200000

alpha=1e-9


sim = quick_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, duration=duration, sampling_rate=sampling_rate)
plot_matrices([sim.X, sim.X_pinv], titles=["X_true", "X_pinv"], show_values=False, dpi=100)
S, N = sim.X.shape

### Random groups

In [ ]:
random_groups_matrix = random_groups(S, N, seed=0)[:3*N]
X_rg = group_lasso_solve(sim, alpha, "random", max_iter=max_iter)
plot_matrices([X_rg, random_groups_matrix], titles=["X (random groups)", "first 3N groups"], show_values=False, dpi=100)

### LASSO with no groups

In [ ]:

no_groups_matrix = no_groups(S, N)[:3*N]#[:S*N].reshape(S,N)
X_none = group_lasso_solve(sim, alpha, grouping="none", max_iter=max_iter)
plot_matrices([X_none, no_groups_matrix], show_values=False, dpi=100)

### Real and imaginary pairs in groups

In [ ]:
real_imag_matrix = real_imag_groups(S, N)[:3*N]#.reshape(S,N)
X_ri = group_lasso_solve(sim, alpha, "real_imag", max_iter=max_iter)
plot_matrices([X_ri, real_imag_matrix], titles=["X_ri", "real_imag_groups"], show_values=False, dpi=100)

### Frequency groups

In [ ]:
frequency_groups_matrix = frequency_groups(S, N)[:3*N]#.reshape(S,N)
X_fg = group_lasso_solve(sim, alpha, "frequency", max_iter=max_iter)
plot_matrices([X_fg, frequency_groups_matrix], titles=["X_fg", "frequency_groups"], show_values=False, dpi=100)

# Arc simulation

In [ ]:
# arc simulation
num_mics=2
num_sources=5
num_active=3
duration=0.1
sampling_rate=2000
mode="sine"


sim = quick_sector_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, duration=duration, sampling_rate=sampling_rate, mode=mode)
plot_signals(ifft(sim.Y))
plt.show()

# Arc with Noise

In [ ]:
snr_db = 20

sim = quick_sector_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, duration=duration, sampling_rate=sampling_rate, mode=mode, snr_db=snr_db)
plot_signals(ifft(sim.Y))
plt.show()

## ⚠️  Pseudoinversen svikter for likt antall mikrofoner som kilder (25,25)

In [ ]:
num_mics=25
num_sources=25
num_active=2
seed=42
mode="sine"
sampling_rate=2000
duration=0.1
source_distance=0.5 # 50 cm
mic_radius = 0.05 # 5 cm

# No noise
sim = quick_sector_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, seed=seed, mode=mode, sampling_rate=sampling_rate, duration=duration, source_distance=source_distance, mic_radius=mic_radius)
print(f"Y: {sim.Y.shape}  A: {sim.A.shape}  X: {sim.X.shape}")

fig, ax = plt.subplots(figsize=(15, 15))
plot_geometry_on_ax(ax, sim.mics, sim.sources, pad_factor=0.1)
ax.set_title("Arc geometry")
fig.dpi= 50
plt.show()

In [ ]:
plot_matrices([sim.X, sim.X_pinv], show_values=False, dpi=100)
plot_recovery(sim, sim.X_pinv, f"$X_{{pinv}}$ for {num_mics} mics and {num_sources} srcs", dpi=100)
plt.show()

In [ ]:
source_distance=0.1 # 10 cm
mic_radius = 0.09 # 5 cm

# No noise
sim = quick_sector_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, seed=seed, mode=mode, sampling_rate=sampling_rate, duration=duration, source_distance=source_distance, mic_radius=mic_radius)

fig, ax = plt.subplots(figsize=(15, 15))
plot_geometry_on_ax(ax, sim.mics, sim.sources, pad_factor=0.1)
ax.set_title("Arc geometry")
fig.dpi= 50
plt.show()

In [ ]:
plot_matrices([sim.X, sim.X_pinv], show_values=False, dpi=100)
plot_recovery(sim, sim.X_pinv, f"$X_{{pinv}}$ for {num_mics} mics and {num_sources} srcs", dpi=100)
plt.show()

# Simulation that I think would work

In [ ]:
num_mics=25
num_sources=25
num_active=2
seed=42
mode="sine"
sampling_rate=2000
duration=0.1
source_distance=0.1 # 10 cm
mic_radius = 0.05 # 5 cm

# No noise
sim = quick_sector_sim(num_mics=num_mics, num_sources=num_sources, num_active=num_active, seed=seed, mode=mode, sampling_rate=sampling_rate, duration=duration, source_distance=source_distance, mic_radius=mic_radius)
print(f"Y: {sim.Y.shape}  A: {sim.A.shape}  X: {sim.X.shape}")

fig, ax = plt.subplots(figsize=(15, 15))
plot_geometry_on_ax(ax, sim.mics, sim.sources, pad_factor=0.1)
ax.set_title("Arc geometry")
fig.dpi= 50
plt.show()

plot_recovery(sim, sim.X_pinv, f"$X_{{pinv}}$ for {num_mics} mics and {num_sources} srcs", dpi=100)
plt.show()

In [ ]:
print(sim.freqs)